In [1]:
%load_ext autoreload
%load_ext nb_black
%autoreload 2

<IPython.core.display.Javascript object>

In [69]:
from copy import deepcopy
from itertools import chain, repeat
from typing import Callable, Iterable, List, Dict, Optional
import sys
import os
import pandas as pd
import torch
from confit import Cli
from pydantic import DirectoryPath
from spacy import displacy
from spacy.tokens import Doc, Span
from tqdm import tqdm
from spacy import Language
from confit.utils.random import set_seed
from transformers import AutoTokenizer, AutoModel
import edsnlp, edsnlp.pipes as eds
from edsnlp import registry, Pipeline
from edsnlp.scorers.ner import create_ner_exact_scorer
from torch import Tensor
import matplotlib.pyplot as plt
from datetime import datetime
import datetime
import time

<IPython.core.display.Javascript object>

In [70]:
dossier = "/home/pidoux/LIMICS/brat/data/RENE"

<IPython.core.display.Javascript object>

In [71]:
doc_iterator = edsnlp.data.read_standoff(
    dossier,
    span_setter={"ents": "Temporal"},
)
true_docs = list(doc_iterator)

2024-04-06 16:35:27.222 | INFO     | edsnlp.data.standoff:__init__:324 - The BRAT directory contains 3 .txt files.


<IPython.core.display.Javascript object>

In [72]:
#displacy.render(true_docs[0], style="ent")

<IPython.core.display.Javascript object>

In [73]:
def txt_liste(dossier:str):
    corpus = []
    for fichier in os.listdir(dossier):
        chemin = os.path.join(dossier, fichier)
        if os.path.isfile(chemin) and fichier.endswith('.txt'):
            with open(chemin, 'r', encoding='utf-8') as f:
                corpus.append(f.read())
    return corpus

<IPython.core.display.Javascript object>

In [102]:
corpus = txt_liste(dossier)
docs = edsnlp.data.from_iterable(corpus)


<IPython.core.display.Javascript object>

In [103]:
nlp = edsnlp.blank("eds")
nlp.add_pipe(eds.sentences())
nlp.add_pipe(eds.normalizer())
nlp.add_pipe(eds.dates()) 
pred_iterator = docs.map_pipeline(nlp)
pred_docs = list(pred_iterator)

<IPython.core.display.Javascript object>

In [107]:
#displacy.render(pred_docs[0], style="span", options={"spans_key": "dates"} )

<IPython.core.display.Javascript object>

In [ ]:
pred_docs[0].ents = pred_docs[0].spans["dates"]
